<a href="https://colab.research.google.com/github/paulheather147/FinalYearProject/blob/main/BinaryBaselineEnsemble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q datasets

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, optimizers
from datasets import load_dataset
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_preprocess
from tensorflow.keras.applications import VGG16, EfficientNetB2

import numpy as np

vgg_img_size = 224
eff_img_size = 260
batch_size = 50
num_classes = 2

In [ ]:
dataset = load_dataset("Falah/Alzheimer_MRI")
print(dataset)
print(dataset["train"].features)

In [ ]:
train_val_split = dataset["train"].train_test_split(test_size=0.2, seed=42)
train = train_val_split["train"]
val = train_val_split["test"]
test = dataset["test"]

def ensure_channel_dim(images):
    current_rank = tf.rank(images)

    def add_channel():
        return tf.expand_dims(images, axis=-1)

    def keep_same():
        return images

    return tf.cond(tf.equal(current_rank, 2), add_channel, keep_same)

def ensure_rgb_channels(images):
    current_channels = tf.shape(images)[-1]

    def to_rgb():
        return tf.image.grayscale_to_rgb(images)

    def drop_alpha():
        return images[..., :3]

    def keep_same():
        return images

    images = tf.cond(tf.equal(current_channels, 1), to_rgb, keep_same)

    new_channels = tf.shape(images)[-1]

    images = tf.cond(tf.equal(new_channels, 4), drop_alpha, keep_same)

    return images

def map_to_binary_label(label_tensor):
    is_non_demented = tf.equal(label_tensor, 2)
    binary_label = tf.where(is_non_demented, 0, 1)
    return tf.cast(binary_label, tf.int32)



def to_tensorflow_dataset(dataset_split, image_size, preprocess_fn, shuffle=False):
    dataset_tf = dataset_split.to_tf_dataset(
        columns=["image", "label"],
        shuffle=shuffle,
        num_workers=0,
    )

    def preprocess_input(example_dict):
        images = tf.cast(example_dict["image"], tf.float32)
        images = ensure_channel_dim(images)
        images = ensure_rgb_channels(images)
        images.set_shape([None, None, 3])

        images = tf.image.resize(images, (image_size, image_size))

        images = preprocess_fn(images)

        original_label = tf.cast(example_dict["label"], tf.int32)
        binary_label = map_to_binary_label(original_label)

        return images, tf.cast(binary_label, tf.float32)

    dataset_tf = dataset_tf.map(preprocess_input,
                                num_parallel_calls=tf.data.AUTOTUNE)
    dataset_tf = dataset_tf.batch(batch_size)
    return dataset_tf.prefetch(tf.data.AUTOTUNE)


vgg_train_ds = to_tensorflow_dataset(train, vgg_img_size, vgg_preprocess, shuffle=True)
vgg_val_ds   = to_tensorflow_dataset(val,   vgg_img_size, vgg_preprocess, shuffle=False)
vgg_test_ds  = to_tensorflow_dataset(test,  vgg_img_size, vgg_preprocess, shuffle=False)


eff_train_ds = to_tensorflow_dataset(train, eff_img_size, eff_preprocess, shuffle=True)
eff_val_ds   = to_tensorflow_dataset(val,   eff_img_size, eff_preprocess, shuffle=False)
eff_test_ds  = to_tensorflow_dataset(test,  eff_img_size, eff_preprocess, shuffle=False)

In [ ]:
vgg_base = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3),
)

vgg_base.trainable = False

vgg_inputs = tf.keras.Input(shape=(224, 224, 3))
x = vgg_base(vgg_inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.5)(x)
vgg_outputs = layers.Dense(1, activation="sigmoid")(x)

vgg_model = tf.keras.Model(vgg_inputs, vgg_outputs)

vgg_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [ ]:
VGG_EPOCHS = 50

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_accuracy',
    factor=0.1,
    patience=5,
    min_lr=1e-6
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=12,
    restore_best_weights=True,
    verbose=1
)

vgg_history_phase1 = vgg_model.fit(
    vgg_train_ds,
    validation_data=vgg_val_ds,
    epochs=VGG_EPOCHS,
    callbacks=[reduce_lr, early_stop],
)

print("\n=== PHASE 2: Fine-tuning top layers ===")
vgg_base.trainable = True
for layer in vgg_base.layers[:-8]:
    layer.trainable = False

print(f"Trainable layers: {sum([l.trainable for l in vgg_base.layers])} / {len(vgg_base.layers)}")

vgg_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

vgg_history_phase2 = vgg_model.fit(
    vgg_train_ds,
    validation_data=vgg_val_ds,
    epochs=VGG_EPOCHS,
    callbacks=[reduce_lr, early_stop],
)

print("\n=== FINAL VGG16 EVALUATION ===")
vgg_test_loss, vgg_test_acc = vgg_model.evaluate(vgg_test_ds, verbose=0)
print(f"VGG16 final test accuracy: {vgg_test_acc:.4f}")


Epoch 1/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 58s 476ms/step - accuracy: 0.5794 - loss: 1.3103 - val_accuracy: 0.7178 - val_loss: 0.5749 - learning_rate: 0.0010
Epoch 2/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 16s 197ms/step - accuracy: 0.6738 - loss: 0.5952 - val_accuracy: 0.7285 - val_loss: 0.5470 - learning_rate: 0.0010
Epoch 3/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 16s 197ms/step - accuracy: 0.6955 - loss: 0.5740 - val_accuracy: 0.7451 - val_loss: 0.5409 - learning_rate: 0.0010
Epoch 4/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 16s 197ms/step - accuracy: 0.7293 - loss: 0.5449 - val_accuracy: 0.7422 - val_loss: 0.5161 - learning_rate: 0.0010
Epoch 5/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 16s 197ms/step - accuracy: 0.7410 - loss: 0.5221 - val_accuracy: 0.7441 - val_loss: 0.4999 - learning_rate: 0.0010
Epoch 6/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 16s 197ms/step - accuracy: 0.7517 - loss: 0.5089 - val_accuracy: 0.7715 - val_loss: 0.4872 - learning_rate: 0.0010
Epoch 7/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 16s 196ms/step - accuracy: 0.7746 - loss: 0.

In [ ]:
import os
save_dir = "/content/drive/MyDrive/alz_models"
os.makedirs(save_dir, exist_ok=True)

vgg_path = os.path.join(save_dir, "BinaryBaselineEnsembleVGG.keras")
vgg_model.save(vgg_path)

print("Saved VGG model to:", vgg_path)

Saved VGG model to: /content/drive/MyDrive/alz_models/BinaryBaselineEnsembleVGG.keras


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_true = []
y_pred = []

for images, labels in vgg_test_ds:
    preds = vgg_model.predict(images)
    y_true.extend(labels.numpy())
    y_pred.extend((preds.flatten() >= 0.5).astype(int))

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))

2/2 ━━━━━━━━━━━━━━━━━━━━ 17s 6s/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 956ms/step
[[622  12]
 

In [ ]:
eff_base = EfficientNetB2(
    weights="imagenet",
    include_top=False,
    input_shape=(260, 260, 3),
)

eff_base.trainable = False

eff_inputs = tf.keras.Input(shape=(260, 260, 3))
y = eff_base(eff_inputs, training=False)
y = layers.GlobalAveragePooling2D()(y)
y = layers.Dense(256, activation="relu")(y)
y = layers.Dropout(0.5)(y)
eff_outputs = layers.Dense(1, activation="sigmoid")(y)

eff_model = tf.keras.Model(eff_inputs, eff_outputs)

eff_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

eff_model.summary()

31790344/31790344 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 260, 260, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb2 (Functional)     │ (None, 9, 9, 1408)     │     7,768,569 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1408)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       360,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,129,530 (31.01 MB)

 Trainable params: 360,961 (1.38 MB)

 Non-trainable params: 7,768,569 (29.63 MB)

In [ ]:
EFF_EPOCHS = 50

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_accuracy',
    factor=0.1,
    patience=5,
    min_lr=1e-6
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=8,
    restore_best_weights=True,
    verbose=1
)

eff_history_phase1 = eff_model.fit(
    eff_train_ds,
    validation_data=eff_val_ds,
    epochs=EFF_EPOCHS,
    callbacks=[reduce_lr, early_stop],
)

print("\n=== PHASE 2: Fine-tuning top layers ===")
eff_base.trainable = True
for layer in eff_base.layers[:-8]:
    layer.trainable = False

print(f"Trainable layers: {sum([l.trainable for l in eff_base.layers])} / {len(eff_base.layers)}")

eff_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

eff_history_phase2 = eff_model.fit(
    eff_train_ds,
    validation_data=eff_val_ds,
    epochs=EFF_EPOCHS,
    callbacks=[reduce_lr, early_stop],
)

print("\n=== FINAL EfficientNetB2 EVALUATION ===")
eff_test_loss, eff_test_acc = eff_model.evaluate(eff_test_ds, verbose=0)
print(f"EfficientNetB2 final test accuracy: {eff_test_acc:.4f}")

Epoch 1/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 117s 885ms/step - accuracy: 0.5950 - loss: 0.6775 - val_accuracy: 0.7109 - val_loss: 0.5628 - learning_rate: 0.0010
Epoch 2/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 14s 167ms/step - accuracy: 0.6797 - loss: 0.5926 - val_accuracy: 0.7334 - val_loss: 0.5550 - learning_rate: 0.0010
Epoch 3/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 14s 166ms/step - accuracy: 0.7039 - loss: 0.5791 - val_accuracy: 0.7197 - val_loss: 0.5394 - learning_rate: 0.0010
Epoch 4/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 14s 167ms/step - accuracy: 0.6720 - loss: 0.5872 - val_accuracy: 0.7402 - val_loss: 0.5358 - learning_rate: 0.0010
Epoch 5/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 14s 165ms/step - accuracy: 0.7000 - loss: 0.5629 - val_accuracy: 0.7197 - val_loss: 0.5449 - learning_rate: 0.0010
Epoch 6/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 14s 166ms/step - accuracy: 0.6920 - loss: 0.5772 - val_accuracy: 0.7158 - val_loss: 0.5494 - learning_rate: 0.0010
Epoch 7/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 14s 166ms/step - accuracy: 0.6897 - loss: 0

In [ ]:
save_dir = "/content/drive/MyDrive/alz_models"
os.makedirs(save_dir, exist_ok=True)

eff_path = os.path.join(save_dir, "BinaryBaselineEnsembleEFF.keras")
eff_model.save(eff_path)

print("Saved EFF model to:", eff_path)

Saved EFF model to: /content/drive/MyDrive/alz_models/BinaryBaselineEnsembleEFF.keras


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_true = []
y_pred = []

for images, labels in eff_test_ds:
    preds = eff_model.predict(images)
    y_true.extend(labels.numpy())
    y_pred.extend((preds.flatten() >= 0.5).astype(int))

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))

2/2 ━━━━━━━━━━━━━━━━━━━━ 43s 21s/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step
[[574  60]
 [ 

In [ ]:
vgg_probs = vgg_model.predict(vgg_test_ds)
eff_probs = eff_model.predict(eff_test_ds)

ensemble_probs = (vgg_probs + eff_probs) / 2.0
ensemble_preds = (ensemble_probs.flatten() >= 0.5).astype(int)

y_true = np.concatenate([y.numpy() for _, y in vgg_test_ds], axis=0)

vgg_preds = (vgg_probs.flatten() >= 0.5).astype(int)
eff_preds = (eff_probs.flatten() >= 0.5).astype(int)

print("VGG16 test accuracy: ", accuracy_score(y_true, vgg_preds))
print("EfficientNetB2 test acc: ", accuracy_score(y_true, eff_preds))
print("Ensemble test accuracy: ", accuracy_score(y_true, ensemble_preds))

print("\nEnsemble classification report:")
print(classification_report(y_true, ensemble_preds, digits=4))

print("\nEnsemble confusion matrix:")
print(confusion_matrix(y_true, ensemble_preds))

26/26 ━━━━━━━━━━━━━━━━━━━━ 6s 187ms/step
26/26 ━━━━━━━━━━━━━━━━━━━━ 21s 473ms/step
VGG16 test accuracy:  0.9796875
EfficientNetB2 test acc:  0.91875
Ensemble test accuracy:  0.97890625

Ensemble classification report:
              precision    recall  f1-score   support

         0.0     0.9810    0.9763    0.9787       634
         1.0     0.9769    0.9814    0.9792       646

    accuracy                         0.9789      1280
   macro avg     0.9789    0.9789    0.9789      1280
weighted avg     0.9789    0.9789    0.9789      1280


Ensemble confusion matrix:
[[619  15]
 [ 12 634]]


In [ ]:
from tensorflow.keras.models import load_model
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from google.colab import drive
import os

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

vgg_path = "/content/drive/MyDrive/alz_models/BinaryBaselineEnsembleVGG.keras"
eff_path = "/content/drive/MyDrive/alz_models/BinaryBaselineEnsembleEFF.keras"

vggmodel = load_model(vgg_path, compile=False)
effmodel = load_model(eff_path, compile=False)

vgg_probs = vggmodel.predict(vgg_test_ds)
eff_probs = effmodel.predict(eff_test_ds)

ensemble_probs = (vgg_probs + eff_probs) / 2.0
ensemble_preds = (ensemble_probs.flatten() >= 0.5).astype(int)

y_true = np.concatenate([y.numpy() for _, y in vgg_test_ds], axis=0)

vgg_preds = (vgg_probs.flatten() >= 0.5).astype(int)
eff_preds = (eff_probs.flatten() >= 0.5).astype(int)

print("VGG16 test accuracy: ", accuracy_score(y_true, vgg_preds))
print("EfficientNetB2 test acc: ", accuracy_score(y_true, eff_preds))
print("Ensemble test accuracy: ", accuracy_score(y_true, ensemble_preds))

print("\nEnsemble classification report:")
print(classification_report(y_true, ensemble_preds, digits=4))

print("\nEnsemble confusion matrix:")
print(confusion_matrix(y_true, ensemble_preds))

26/26 ━━━━━━━━━━━━━━━━━━━━ 6s 185ms/step
26/26 ━━━━━━━━━━━━━━━━━━━━ 21s 476ms/step
VGG16 test accuracy:  0.9796875
EfficientNetB2 test acc:  0.91875
Ensemble test accuracy:  0.97890625

Ensemble classification report:
              precision    recall  f1-score   support

         0.0     0.9810    0.9763    0.9787       634
         1.0     0.9769    0.9814    0.9792       646

    accuracy                         0.9789      1280
   macro avg     0.9789    0.9789    0.9789      1280
weighted avg     0.9789    0.9789    0.9789      1280


Ensemble confusion matrix:
[[619  15]
 [ 12 634]]
